# LangChain How To Start
https://www.langchain.com.cn/docs/how_to/

# 安装Langchain

详细内容见：
[LangChain](https://www.langchain.com.cn/docs/how_to/installation/)

`pip install langchain`

![Langchain](../img/Langchain-communication.png)

- 除了 `langsmith SDK`，LangChain 生态系统中的所有包都依赖于 `langchain-core`
- 某些集成，如 OpenAI 和 Anthropic，有自己的包。[集成文档](https://www.langchain.com.cn/docs/integrations/platforms/)
    - `pip install langchain-openai`
- 未拆分为自己包的集成将保留在 `langchain-community` 包中
    - `pip install langchain-community`

# 主要特性
- 这突出了使用LangChain的核心功能。
    - 从模型**返回结构化数据**
    - 使用模型**调用工具**
    - **流式**运行
    - **调试**你的大型语言模型**应用**

1. 返回机构化数据 `.with_structured_output()`

In [1]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")

SECESSFULLY!


In [2]:
from langchain_deepseek import ChatDeepSeek
ChatDeepSeek?

Init signature:
ChatDeepSeek(
    *args: Any,
    name: Optional[str] = None,
    cache: Union[langchain_core.caches.BaseCache, bool, NoneType] = None,
    verbose: bool = <factory>,
    callbacks: Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType] = None,
    tags: Optional[list[str]] = None,
    metadata: Optional[dict[str, Any]] = None,
    custom_get_token_ids: Optional[Callable[[str], list[int]]] = None,
    callback_manager: Optional[langchain_core.callbacks.base.BaseCallbackManager] = None,
    rate_limiter: Optional[langchain_core.rate_limiters.BaseRateLimiter] = None,
    disable_streaming: Union[bool, Literal['tool_calling']] = False,
    client: Any = None,
    async_client: Any = None,
    root_client: Any = None,
    root_async_client: Any = None,
    model: str,
    temperature: Optional[float] = None,
    model_kwargs: dict[str, typing.Any] = <factory>,
    api_key: Optional[pydantic.types.SecretStr

In [3]:
llm = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)

## Pydantic 类
使用 Pydantic 的主要优点是模型生成的输出将会被验证。如果缺少任何必需字段或字段类型错误，Pydantic 将引发错误。

In [4]:
from typing import Optional
from pydantic import BaseModel, Field

# Pydantic
class Joke(BaseModel):
    """说一个冷笑话"""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")
    rating: int = Field(
        default=0, description="给笑话的好笑程度从1-10打分"
    )
structured_llm = llm.with_structured_output(Joke)
structured_llm.invoke("给我将一个关于猫和老鼠的笑话")

Joke(setup='一只猫和一只老鼠在厨房相遇了', punchline="老鼠对猫说：'你抓不到我，因为我是无线鼠标！'", rating=7)

### TypedDict 或 JSON Schema
- 如果您不想使用 Pydantic，明确不想对参数进行验证，
- 或者希望能够流式处理模型输出，您可以使用 TypedDict 类定义您的模式。

我们可以选择性地使用 LangChain 支持的特殊 Annotated 语法，允许您指定字段的默认值和描述。请注意，如果模型没有生成默认值，则默认值不会自动填充，它仅用于定义传递给模型的模式。

In [5]:
"""核心: langchain-core>=0.2.26
类型扩展: 强烈建议从 typing_extensions 导入 Annotated 和 TypedDict，
        而不是从 typing 导入，以确保在不同 Python 版本之间的一致行为。"""

from typing_extensions import Annotated, TypedDict

# TypedDict
class Joke(TypedDict):
    """Joke to tell user."""

    setup: Annotated[str, ..., "The setup of the joke"]

    # Alternatively, we could have specified setup as:

    # setup: str                    # no default, no description
    # setup: Annotated[str, ...]    # no default, no description
    # setup: Annotated[str, "foo"]  # default, no description

    punchline: Annotated[str, ..., "The punchline of the joke"]
    rating: Annotated[Optional[int], None, "How funny the joke is, from 1 to 10"]


structured_llm = llm.with_structured_output(Joke)
structured_llm.invoke("讲一个关于明天的有趣的黑笑话")

{'setup': '为什么程序员总是对明天充满期待？', 'punchline': '因为他们知道明天会有更多的bug要修复！', 'rating': 7}

同样，我们可以传入一个 ``JSON Schema`` 字典。

In [6]:
json_schema = {
    "title": "joke",
    "description": "Joke to tell user.",
    "type": "object",
    "properties": {
        "setup": {
            "type": "string",
            "description": "The setup of the joke",
        },
        "punchline": {
            "type": "string",
            "description": "The punchline to the joke",
        },
        "rating": {
            "type": "integer",
            "description": "How funny the joke is, from 1 to 10",
            "default": None,
        },
    },
    "required": ["setup", "punchline"],
}

structured_llm = llm.with_structured_output(json_schema)
structured_llm.invoke("Tell me a joke about football")

{'setup': 'Why did the football coach go to the bank?',
 'punchline': 'To get his quarter back!',
 'rating': 7}

## 在多个模式选择
让模型从多个模式中选择的最简单方法是创建一个具有联合类型属性的父模式：

In [7]:
from typing import Union

# Pydantic
class Joke(BaseModel):
    """Joke to tell user."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")
    rating: Optional[int] = Field(
        default=None, description="How funny the joke is, from 1 to 10"
    )


class ConversationalResponse(BaseModel):
    """Respond in a conversational manner. Be kind and helpful."""

    response: str = Field(description="A conversational response to the user's query")


class FinalResponse(BaseModel):
    final_output: Union[Joke, ConversationalResponse]


structured_llm = llm.with_structured_output(FinalResponse)

structured_llm.invoke("你今天怎么样?")

FinalResponse(final_output=ConversationalResponse(response='谢谢你的关心！作为一个AI助手，我没有情感状态，但我的系统运行得很顺畅，随时准备为你提供帮助。你今天过得怎么样？有什么我可以为你做的吗？'))

In [8]:
class FinalResponse(BaseModel):
    final_output: Joke | ConversationalResponse

structured_llm = llm.with_structured_output(FinalResponse)
structured_llm.invoke("说一个超级无敌大爆的笑话！")

FinalResponse(final_output=Joke(setup='为什么鸡要过马路？', punchline='因为它要去对面的KFC面试！', rating=10))

## 流式处理
当模式被指定为TypedDict类或JSON Schema字典时，我们可以从我们的结构化模型中流式输出。

In [9]:
from typing_extensions import Annotated, TypedDict


# TypedDict
class Joke(TypedDict):
    """Joke to tell user."""

    setup: Annotated[str, ..., "The setup of the joke"]
    punchline: Annotated[str, ..., "The punchline of the joke"]
    rating: Annotated[Optional[int], None, "How funny the joke is, from 1 to 10"]


structured_llm = llm.with_structured_output(Joke)

for chunk in structured_llm.stream("说一个超级无敌大爆的冷笑话！"):
    print(chunk)

{}
{'setup': ''}
{'setup': '为什么'}
{'setup': '为什么企'}
{'setup': '为什么企鹅'}
{'setup': '为什么企鹅的'}
{'setup': '为什么企鹅的肚子'}
{'setup': '为什么企鹅的肚子是'}
{'setup': '为什么企鹅的肚子是白色的'}
{'setup': '为什么企鹅的肚子是白色的？'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': ''}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱太久'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱太久，'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱太久，你的'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱太久，你的手'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱太久，你的手也会'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱太久，你的手也会变'}
{'setup': '为什么企鹅的肚子是白色的？', 'punchline': '因为如果你把手伸进冰箱太久，你的手也会变白'}
{'setup': '为什么企鹅的肚子是白色的？', 'punch

## 少量示例提示
对于更复杂的模式，将少量示例添加到提示中非常有用。

1. 最简单和最通用的方法是将示例添加到提示中的系统消息中
["ChatPromptTemplate--docs"]("https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html")

In [10]:
from langchain_core.prompts import ChatPromptTemplate

system = """You are a hilarious comedian. Your specialty is knock-knock jokes. \
Return a joke which has the setup (the response to "Who's there?") and the final punchline (the response to "<setup> who?").

Here are some examples of jokes:

example_user: Tell me a joke about planes
example_assistant: {{"setup": "Why don't planes ever get tired?", "punchline": "Because they have rest wings!", "rating": 2}}

example_user: Tell me another joke about planes
example_assistant: {{"setup": "Cargo", "punchline": "Cargo 'vroom vroom', but planes go 'zoom zoom'!", "rating": 10}}

example_user: Now about caterpillars
example_assistant: {{"setup": "Caterpillar", "punchline": "Caterpillar really slow, but watch me turn into a butterfly and steal the show!", "rating": 5}}"""

prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{input}")])

few_shot_structured_llm = prompt | structured_llm
few_shot_structured_llm.invoke("what's something funny about woodpeckers")

{'setup': 'Woodpecker',
 'punchline': 'Woodpecker just trying to find the right spot to hang my pictures!',
 'rating': 7}

当结构化输出的底层方法是工具调用时，我们可以将示例作为显式工具调用传入。

[AIMessage](https://python.langchain.com/api_reference/core/messages/langchain_core.messages.ai.AIMessage.html) |
[HumanMessage](https://python.langchain.com/api_reference/core/messages/langchain_core.messages.human.HumanMessage.html) |
[ToolMessage](https://python.langchain.com/api_reference/core/messages/langchain_core.messages.tool.ToolMessage.html)
<!--IMPORTS:[{"imported": "", "source": "langchain_core.messages", "docs": "", "title": "How to return structured data from a model"}, {"imported": "HumanMessage", "source": "langchain_core.messages", "docs": "", "title": "How to return structured data from a model"}, {"imported": "ToolMessage", "source": "langchain_core.messages", "docs": "", "title": "How to return structured data from a model"}]-->

In [14]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from typing_extensions import Annotated, TypedDict


# TypedDict
class Joke(TypedDict):
    """Joke to tell user."""

    setup: Annotated[str, ..., "The setup of the joke"]
    punchline: Annotated[str, ..., "The punchline of the joke"]
    rating: Annotated[Optional[int], None, "How funny the joke is, from 1 to 10"]


structured_llm = llm.with_structured_output(Joke)

examples = [
    HumanMessage("Tell me a joke about planes", name="example_user"),
    AIMessage(
        "",
        name="example_assistant",
        tool_calls=[
            {
                "name": "joke",
                "args": {
                    "setup": "Why don't planes ever get tired?",
                    "punchline": "Because they have rest wings!",
                    "rating": 2,
                },
                "id": "1",
            }
        ],
    ),
    # Most tool-calling models expect a ToolMessage(s) to follow an AIMessage with tool calls.
    ToolMessage("", tool_call_id="1"),
    # Some models also expect an AIMessage to follow any ToolMessages,
    # so you may need to add an AIMessage here.
    HumanMessage("Tell me another joke about planes", name="example_user"),
    AIMessage(
        "",
        name="example_assistant",
        tool_calls=[
            {
                "name": "joke",
                "args": {
                    "setup": "Cargo",
                    "punchline": "Cargo 'vroom vroom', but planes go 'zoom zoom'!",
                    "rating": 10,
                },
                "id": "2",
            }
        ],
    ),
    ToolMessage("", tool_call_id="2"),
    HumanMessage("Now about caterpillars", name="example_user"),
    AIMessage(
        "",
        tool_calls=[
            {
                "name": "joke",
                "args": {
                    "setup": "Caterpillar",
                    "punchline": "Caterpillar really slow, but watch me turn into a butterfly and steal the show!",
                    "rating": 5,
                },
                "id": "3",
            }
        ],
    ),
    ToolMessage("", tool_call_id="3"),
]
system = """You are a hilarious comedian. Your specialty is knock-knock jokes. \
Return a joke which has the setup (the response to "Who's there?") \
and the final punchline (the response to "<setup> who?")."""

prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("placeholder", "{examples}"), ("human", "{input}")]
)
few_shot_structured_llm = prompt | structured_llm
result = few_shot_structured_llm.invoke({"input": "晴天", "examples": examples})
print(result)

None


## 指定结构化输出
对于支持多种结构化输出方式的模型，使用 `method=` 指定

In [12]:
structured_llm_json = llm.with_structured_output(None, method="json_mode")

structured_llm_json.invoke(
    "Tell me a joke about cats, respond in JSON with `setup` and `punchline` keys"
)

{'setup': "Why don't cats play poker in the jungle?",
 'punchline': 'Too many cheetahs!'}

### 原始输出
大型语言模型在生成结构化输出方面并不完美，尤其是当模式变得复杂时。您可以通过传递 `include_raw=True` 来避免引发异常并自行处理原始输出。
- 包含原始消息输出、parsed 值（如果成功）以及任何结果错误

In [17]:
structured_llm = llm.with_structured_output(Joke, include_raw=True)

structured_llm.invoke("Tell me a joke about cats")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_00_OFcgFwfYnlAESyqvnHWomD5F', 'function': {'arguments': '{"setup": "Why don\'t cats play poker in the jungle?", "punchline": "Too many cheetahs!", "rating": 7}', 'name': 'Joke'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 214, 'total_tokens': 252, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 22}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'cc062544-8087-41a6-bcc2-56f6b57b4126', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--68b9c426-1c4a-4a39-9e9e-685d7cc24f5e-0', tool_calls=[{'name': 'Joke', 'args': {'setup': "Why don't cats play poker in the jungle?", 'punchline': 'Too many cheetahs!', 'rating': 7}, 'id': 'call_00_OFcg

# 直接提示和解析模型输出
并非所有模型都支持 .with_structured_output()，因为并非所有模型都具有工具调用或 JSON 模式支持。
- 使用 [PydanticOutputParser](https://python.langchain.com/api_reference/core/output_parsers/langchain_core.output_parsers.pydantic.PydanticOutputParser.html) 来解析被提示以匹配给定 Pydantic 模式的聊天模型的输出。

In [18]:
from typing import List

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class Person(BaseModel):
    """Information about a person."""

    name: str = Field(..., description="The name of the person")
    height_in_meters: float = Field(
        ..., description="The height of the person expressed in meters."
    )


class People(BaseModel):
    """Identifying information about all people in a text."""
    people: List[Person]


# Set up a parser
parser = PydanticOutputParser(pydantic_object=People)

# Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the user query. Wrap the output in `json` tags\n{format_instructions}",
        ),
        ("human", "{query}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

In [22]:
query = "Anna is 32 years old and she is 6 feet tall"

print(prompt.invoke(query).to_string())

chain = prompt | llm | parser

chain.invoke({"query": query})

System: Answer the user query. Wrap the output in `json` tags
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Person": {"description": "Information about a person.", "properties": {"name": {"description": "The name of the person", "title": "Name", "type": "string"}, "height_in_meters": {"description": "The height of the person expressed in meters.", "title": "Height In Meters", "type": "number"}}, "required": ["name", "height_in_meters"], "title": "Person", "type": "object"}}, "description": "Identifying information about all people in a text.", "properties": {"people": {"items"

People(people=[Person(name='Anna', height_in_meters=1.8288)])

自定义解析
- 还可以使用 LangChain表达式 (LCEL) 创建自定义提示和解析器，使用普通函数解析模型的输出：